In [1]:

import argparse
import logging
from types import SimpleNamespace
import os
from pathlib import Path
import csv
import sys
import math
import numpy as np
import pandas as pd
import time
import joblib
import json 
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
# sys.path.append('/Users/yiqin/Documents/PythonProjects/GraspDataProcessing/src')
# sys.path.append('D:\\PythonPrograms\\GraspDataProcessing\\src')
sys.path.append('D:\\PythonProjects\\GraspDataProcessing\\src')
import graspdataprocessing as gdp


In [2]:
config_file = 'config.toml'  # 配置文件的路径
config = gdp.load_config(config_file)

配置验证通过: cutoff_value=1e-09, initial_ratio=0.09


In [3]:
"""主程序逻辑"""
config.file_name = f'{config.conf}_{config.cal_loop_num}'
logger = gdp.setup_logging(config)
logger.info("机器学习训练程序启动")
execution_time = time.time()

gdp.setup_directories(config)
# 初始化结果文件
# gdp.initialize_results_file(config, logger)

# 验证初始文件
gdp.validate_initial_files(config, logger)

logger.info(f"初始比例: {config.initial_ratio}")
logger.info(f"光谱项: {config.spetral_term}")

2025-06-19 10:30:04,411 - INFO - 机器学习训练程序启动
2025-06-19 10:30:04,412 - INFO - 成功加载初始CSFs文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4.c
2025-06-19 10:30:04,412 - INFO - 初始比例: 0.09
2025-06-19 10:30:04,412 - INFO - 光谱项: ['5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_9D', '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_7D']


In [17]:
try:
    # 加载数据文件
    energy_level_data_pd, rmix_file_data, target_pool_csfs_data, raw_csfs_descriptors, cal_csfs_data, caled_csfs_indices_dict, unselected_csfs_indices_dict = gdp.load_data_files(config, logger)
    
    # 检查组态耦合
    cal_result, asfs_position = gdp.check_configuration_coupling(config, energy_level_data_pd, logger)
    logger.info("************************************************")

except Exception as e:
        logger.error(f"程序执行过程中发生错误: {str(e)}")
        raise

2025-06-19 10:35:07,618 - INFO - 加载能级数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level


Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.level loaded.
data file type: level data
energy levels data has been written into LEVEL/cv4odd1as3_odd4_1.level_level.csv csv file
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m loaded.
data file type: rmcdhf mix_coefficient_data
g92mix: G92MIX
 nblock = 1,       ncftot =   385600,          nw =  41,            nelec =   64


100%|██████████| 1/1 [00:00<00:00, 499.44it/s]
2025-06-19 10:35:07,621 - INFO - 加载 *.m 文件数据: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.m


cycle jblock = 1
 Block no. = 1, 2J+1 = 9, Parity = -1, No. of eigenvalues = 2, No. of CSFs = 385600

    Energy levels for ...
Rydberg constant is  109737.31568508

---------------------------------------------
 No Pos  J  Parity   Energy Total    Levels
                      (a.u.)         (cm^-1)
---------------------------------------------

  1  1   4   +    -11274.7413433        0.00
  2  0   4   +    -11274.7204780     4579.40


2025-06-19 10:35:13,475 - INFO - 加载初始 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4.pkl.gz
2025-06-19 10:35:13,704 - INFO - 加载初始 CSFs 描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4


Descriptors loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors.npy
Labels loaded from: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_descriptors_block_indices.npy
Data file D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c loaded.


2025-06-19 10:35:14,137 - INFO - 加载本轮计算 CSFs 文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c
2025-06-19 10:35:14,145 - INFO - 加载本轮选择的 CSFs 的索引文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_chosen_indices.pkl
2025-06-19 10:35:14,146 - INFO - 加载本轮选择的 CSFs 的索引文件: {0: array([ 720050, 1200405,  277663, ..., 1661771, 2738838, 3962938])}
2025-06-19 10:35:14,542 - INFO - 光谱项 '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_9D' 在位置 0 找到
2025-06-19 10:35:14,543 - INFO - 光谱项 '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_7D' 在位置 1 找到
2025-06-19 10:35:14,543 - INFO - cal_loop 1 组态耦合正确，位置索引: [0, 1]
2025-06-19 10:35:14,544 - INFO - ************************************************


In [5]:
logger.info("             数据预处理")
caled_csfs_descriptors = gdp.generate_chosen_csfs_descriptors(config, caled_csfs_indices_dict, raw_csfs_descriptors, rmix_file_data, asfs_position, logger)
unselected_csfs_descriptors = gdp.get_unselected_descriptors(raw_csfs_descriptors, caled_csfs_indices_dict)
X_unselected = unselected_csfs_descriptors.copy()
logger.info("             特征提取完成")

2025-06-19 10:31:14,436 - INFO -              数据预处理


2025-06-19 10:31:14,602 - INFO - 保存本轮选择的 CSFs 的描述符文件: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1/cv4odd1as3_odd4_1.npy


Descriptors saved to: D:\PythonProjects\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1_descriptors.npy


2025-06-19 10:31:15,434 - INFO -              特征提取完成


In [6]:
model, X_train, X_test, y_train, y_test, training_time, weight = gdp.train_model(config, caled_csfs_descriptors, rmix_file_data, logger)

2025-06-19 10:31:16,836 - INFO - 训练集 - 正样本:22038, 负样本:286442, 比例:0.0714
2025-06-19 10:31:17,776 - INFO - 创建新模型，设置类别权重: 负样本=1.0, 正样本=13.0
2025-06-19 10:31:17,776 - INFO - 使用原始数据训练 - 不进行重采样
2025-06-19 10:31:17,777 - INFO - 最终训练数据 - 正样本:22038, 负样本:286442, 比例:0.0714
2025-06-19 10:31:17,777 - INFO - 使用类别权重和损失函数来处理数据不平衡问题
2025-06-19 10:31:17,778 - INFO -              训练模型
2025-06-19 10:31:17,778 - INFO - 开始训练 - 数据量:308,480, 特征维度:87
2025-06-19 10:31:33,063 - INFO - Epoch [50/150], Loss: 0.328800
2025-06-19 10:31:47,324 - INFO - Epoch [100/150], Loss: 0.315715
2025-06-19 10:32:02,266 - INFO - Epoch [150/150], Loss: 0.308631
2025-06-19 10:32:02,266 - INFO - 训练Loss良好 (0.310)，模型收敛效果理想
2025-06-19 10:32:02,267 - INFO -              预测与评估
2025-06-19 10:32:02,359 - INFO - 预测概率统计 - 最小值:0.0001, 最大值:1.0000, 平均值:0.2646
2025-06-19 10:32:02,360 - INFO - 预测为正类的样本数: 5723/77120
2025-06-19 10:32:02,360 - INFO - 真实正样本数: 5423.0/77120
2025-06-19 10:32:02,361 - INFO - 真实正样本比例: 0.070, 预测正样本比例: 0.074


(385600,)


2025-06-19 10:32:03,884 - INFO - 测试集预测结果:
2025-06-19 10:32:03,884 - INFO - AUC:0.8800573070593959, pr_auc:0.762717371891902, f1:0.7416113403911717, accuracy:0.9626556016597511, precision:0.7221736851301764, recall:0.7621242854508574
2025-06-19 10:32:03,983 - INFO - 训练集预测结果:
2025-06-19 10:32:03,984 - INFO - AUC:0.9446302822976329, f1:0.7639467760269241, accuracy:0.9656671421161825, precision:0.7507118139208901, recall:0.7776567746619475


In [7]:
evaluation_results = gdp.evaluate_model(
            model, X_train, X_test, y_train, y_test, X_unselected, config, logger
        )

2025-06-19 10:32:31,787 - INFO - 开始预测与评估
2025-06-19 10:32:32,647 - INFO - 模型评估完成


In [8]:
test_predictions = evaluation_results['predictions']['y_prediction_test']
test_probabilities = evaluation_results['probabilities']['y_probability_test']
test_f1 = evaluation_results['test_metrics']['f1']
train_f1 = evaluation_results['train_metrics']['f1']

In [9]:
overfitting_check = test_f1 - train_f1  # 如果差异过大说明过拟合

In [10]:
overfitting_check

-0.022335435635752354

In [11]:
logger.info("             模型推理")
start_time = time.time()
X_unselected = unselected_csfs_descriptors.copy()
y_unselected_prediction = model.predict(X_unselected)
y_unselected_probability = model.predict_probability(X_unselected)[:, 1]
eval_time = time.time() - start_time
logger.info(f"             模型推理时间:{eval_time}")

2025-06-19 10:32:37,727 - INFO -              模型推理
2025-06-19 10:32:38,608 - INFO -              模型推理时间:0.8809666633605957


In [12]:
y_prediction = evaluation_results['predictions']['y_prediction_test']
y_probability = evaluation_results['probabilities']['y_probability_test']
y_probability_all = evaluation_results['probabilities']['y_probability_all']
y_probability_other = evaluation_results['probabilities']['y_probability_other']
result_file_path = config.root_path / 'test_data' / f'{config.conf}_{config.cal_loop_num}.csv'
pd.DataFrame({"y_test": y_test, "y_prediction": y_prediction, "y_probability": y_probability}).to_csv(result_file_path, index=False)
model_file_path = config.root_path / 'models' / f'{config.conf}_{config.cal_loop_num}.pkl'
joblib.dump(model, model_file_path)
logger.info(f"             预测结果与模型保存成功")

csfs_above_threshold_indices = np.where(np.any(rmix_file_data.mix_coefficient_List[0][asfs_position]**2 >= np.float64(config.cutoff_value), axis = 0))[0]


2025-06-19 10:32:40,271 - INFO -              预测结果与模型保存成功


In [13]:
high_prob_threshold = np.percentile(y_probability_all[:, 1], 90)  # 取90分位数作为高概率阈值

In [14]:
high_prob_threshold

np.float32(0.3764029)

In [26]:
# 一行完成转换和索引, 已经将存储的索引形式改为numpy数组了，这里后续还是改为
promising_unselected_CSFs_indices = unselected_csfs_indices_dict[0][y_probability_other > high_prob_threshold]


In [27]:
promising_unselected_CSFs_indices.shape

(136822,)

In [28]:
filtered_chosen_indices = caled_csfs_indices_dict[0][csfs_above_threshold_indices]

In [29]:
filtered_chosen_indices.shape

(26971,)

In [30]:
all_chosen_indices = np.union1d(filtered_chosen_indices, promising_unselected_CSFs_indices)

In [32]:
all_chosen_indices.shape

(163793,)

In [33]:
ml_chosen_indices_dict = {0 : all_chosen_indices}

In [34]:
ml_chosen_indices_dict

{0: array([      6,      19,      36, ..., 4284073, 4284074, 4284112])}